Basic file processing

A file can be opened with the open function.

The call ```open(filename, mode="r")``` will return a file object, whose type is ```file```.

This file object can be used to refer to a file on disk. For example, when we want to read from or write to a file, we can use the methods ```read``` and ```write``` of the file object.

After the file object is no longer needed, a call to the ```close``` method should be made.

We can control what kind of operations we can perform on a file with the mode parameter of the ```open``` function

 Different options include opening a file for reading or writing, whether the file exists already or needs be created with the call to the open method, etc. Here's a list of all the opening modes:

<div style="max-width: 600px;">

| Mode | Description |
| :--- | :--- |
| `r` | read-only mode, file must exist |
| `w` | write-only mode, creates, or overwrites an existing file |
| `a` | write-only mode, write always appends to the end |
| `r+` | read/write mode, file must already exist |
| `w+` | read/write mode, creates, or overwrites an existing file |
| `a+` | read/write mode, write will append to end |

</div>


In the end of the mode string either the letter ```t``` or ```b``` can be appended. These stand for text mode and binary mode. If this letter is not given, the file type is text mode by default.

## Text Mode vs Binary Mode — Simply

This is about **what happens to the raw bytes** as they travel between the file on disk and your Python program.

---

### The Two Suffixes — `t` and `b`

```python
open("data.txt", "rt")   # explicit text mode
open("data.txt", "r")    # same thing — 't' is the DEFAULT if omitted
open("data.bin", "rb")   # binary mode
```

| Suffix | Mode | Default? |
|---|---|---|
| `t` (or nothing) | **text** mode | ✓ yes, if you omit the letter |
| `b` | **binary** mode | must be explicit |

So `open("f.txt", "r")` and `open("f.txt", "rt")` are **identical** — you've been using text mode all along without realizing it had a name.

---

### Binary Mode — "Don't Touch Anything, Just Give Me the Raw Bytes"

> *"the contents of the file are not interpreted in any way"*

Binary mode hands you the file's contents **exactly as stored** — as raw bytes, no translation, no cleverness:

```python
with open("photo.jpg", "rb") as f:
    data = f.read()
    print(type(data))    # → <class 'bytes'>
```

Remember bytes from way back? *"A byte consists of 8 bits, representing 0–255."* Binary mode gives you the file as a straight sequence of these numbers — like a photo, video, or any file that isn't meant to be read as text. Nothing gets reinterpreted as characters.

---

### Text Mode — Two Automatic Conversions Happen

Text mode is "smarter" — it assumes the file contains **human-readable text**, and does two translations for your convenience:

---

### Conversion 1 — Line Endings (Windows Quirk)

Different operating systems mark "end of line" differently:

```
Windows:      \r\n   (two characters: carriage return + newline)
Linux/Mac:    \n     (one character)
```

Text mode **hides this difference** from you:

```
READING a Windows file:
  On disk:        "Hello\r\nWorld"
  Python gives you: "Hello\nWorld"        ← \r\n silently became \n

WRITING on Windows:
  You write:       "Hello\nWorld"
  Saved to disk as: "Hello\r\nWorld"      ← \n silently became \r\n
```

So no matter which OS created the file, or which OS you're running Python on, **you always work with the simple `'\n'`** inside your program. Text mode does the messy `\r\n ↔ \n` juggling behind the scenes.

**Binary mode skips this entirely** — if the file has `\r\n`, you get exactly `\r\n`, untouched:

```python
with open("winfile.txt", "rb") as f:
    data = f.read()
    print(data)   # → b'Hello\r\nWorld'    the \r survives!
```

---

### Conversion 2 — Bytes ↔ Characters (Encoding)

This is your **UTF-8 knowledge from earlier in this conversation**, applied directly here!

Remember: `"ä".encode('utf-8')` turns one character into multiple bytes. On disk, files store only **bytes** — never actual Python string objects. So reading a text file means converting those raw bytes back into characters:

```
On disk (bytes):     b'\xc3\xa4'
                            ↓  decode using an encoding (e.g. utf-8)
In Python (text mode): 'ä'          ← a proper string character
```

```python
with open("finnish.txt", "rt", encoding="utf-8") as f:
    text = f.read()
    print(type(text))   # → <class 'str'>
```

**Binary mode skips this too** — you'd get the raw bytes `b'\xc3\xa4'`, not the character `'ä'`.

---

### Side-by-Side Comparison

| | Text mode (`t`) | Binary mode (`b`) |
|---|---|---|
| Return type | `str` | `bytes` |
| Line endings | `\r\n` auto-converted to `\n` | left exactly as-is |
| Character encoding | bytes auto-decoded to characters (e.g. via utf-8) | left as raw bytes |
| Good for | `.txt`, `.py`, `.csv`, `.json` | `.jpg`, `.mp3`, `.zip`, `.exe` |

---

### The One-Sentence Summary

> **Text mode** treats the file as human-readable text and quietly handles two annoying details for you — line-ending differences between operating systems, and the byte↔character encoding conversion (like UTF-8). **Binary mode** disables both conversions and gives you the file's exact raw bytes, untouched — correct for anything that isn't meant to be read as text. Text mode is the default if you don't write `t` or `b` explicitly. 🎯

Q: READING a Windows file: On disk: "Hello\r\nWorld" Python gives you: "Hello\nWorld" ← \r\n silently became \n

What is the \r in here

## What Is `\r`?

`\r` is the **carriage return** character — one of the invisible "control characters" you learned about way back when we discussed ASCII (remember `\n`, tab, and other control characters?).

---

### Where the Name Comes From — Typewriters! ⌨️

The name is a leftover from **mechanical typewriters**, decades before computers existed:

```
Typewriters had a physical "carriage" — the part holding the paper
that moved left-to-right as you typed each letter.

To start a new line, you needed TWO physical actions:
  1. "Carriage return"  →  slide the carriage back to the LEFT margin
  2. "Line feed"        →  roll the paper UP one line
```

Two separate mechanical motions. Early computer terminals mimicked typewriters, so they kept both concepts as two separate characters:

```
\r  =  Carriage Return   =  "move cursor back to the start of the line"
\n  =  Line Feed         =  "move down one line"
```

---

### Visualizing What Each One Does

Imagine a text cursor `|`:

```
Before:   Hello|
```

**`\r` alone** — cursor jumps back to column 0, **same line**:
```
Hello\r    →    |Hello        ← cursor is now back at the START,
                                 about to overwrite from the left!
```

**`\n` alone** — cursor drops to the **next line**, same column:
```
Hello\n    →    Hello
                |             ← new line, cursor still under the 'H' position
```

**`\r\n` together (Windows style)** — both happen: drop a line, AND return to column 0:
```
Hello\r\n  →    Hello
                |             ← new line AND at the very start — a proper "new line"
```

---

### Why Windows Uses Both, But Linux/Mac Use Just `\n`

- **Windows** kept the old two-character tradition: `\r\n`
- **Linux/Mac (Unix-family)** simplified it: `\n` alone is defined to mean *"both"* — one character does the whole job

Neither is "wrong" — they're just two different historical conventions for the same real-world action: *"start a fresh line."*

---

### Why This Matters for File Reading

Going back to your original question:

```
On a Windows-created file, on disk:
    "Hello\r\nWorld"
             ↑↑
          BOTH characters are stored — carriage return then line feed
```

When Python opens this in **text mode**, it recognizes the Windows-style pair `\r\n` and **collapses it into the single `\n`** that Python (following the Unix convention) uses internally:

```python
with open("windows_file.txt", "r") as f:
    text = f.read()
    print(text)         # → "Hello\nWorld"   ← \r is gone, just \n remains
```

If you instead read it in **binary mode**, you'd see the raw, untouched pair:

```python
with open("windows_file.txt", "rb") as f:
    data = f.read()
    print(data)          # → b'Hello\r\nWorld'   ← \r survives, nothing was converted
```

---

### See `\r` in Action Yourself

```python
print("Hello\rWorld")
```

Depending on your terminal, this can look surprising — `\r` sends the cursor back to the start of the line, so `World` may **overwrite** `Hello`:

```
World
```

(The `H`, `e`, `l`, `l`, `o` get overwritten by `W`, `o`, `r`, `l`, `d` — same length, so it looks like a clean replace. This trick is actually used deliberately for progress bars in terminals — `print(f"\rProgress: {i}%", end="")` updates the same line repeatedly instead of printing a new line each time.)

---

### The One-Sentence Summary

> `\r` = **carriage return**, a leftover typewriter concept meaning *"move back to the start of the line"* (as opposed to `\n`, "line feed," meaning *"move down one line"*). Windows files store newlines as **both** together (`\r\n`); Python's text mode auto-strips the `\r` so you always work with the clean, cross-platform `\n`. 🎯

Q:If you instead read it in binary mode
with open("windows_file.txt", "rb") as f:
    data = f.read()
    print(data)          # → b'Hello\r\nWorld'   ← \r survives, nothing was converted

This is really confusing right? Because  binary mode means 0 and 1 But in here it just the Text with \r and \n chanraters .

## Excellent Catch — This Really Does Look Contradictory!

You're absolutely right to be confused — the *display* looks like text, but binary mode is genuinely giving you raw bytes. The confusion is entirely about how Python **prints** bytes for you to read, not about what's actually stored. Let me untangle this.

---

### What's ACTUALLY Happening — Two Different Things

**1. What binary mode gives you:** a `bytes` object — a sequence of raw numbers (0-255), exactly as you learned:

```python
data = b'Hello\r\nWorld'
print(type(data))    # → <class 'bytes'>
```

**2. What you're SEEING on screen:** Python's chosen way to **display** a bytes object to a human, so you don't have to stare at raw numbers.

These are different layers — the object itself vs. how it's shown to you.

---

### Proving It's Really Just Numbers Underneath

Let's peel back Python's display trick and look at the **actual bytes**:

```python
data = b'Hello\r\nWorld'
print(list(data))
```

Output:
```
[72, 101, 108, 108, 111, 13, 10, 87, 111, 114, 108, 100]
```

**There it is** — plain numbers, 0-255, exactly like the bytes lesson from way earlier in our conversation! No letters, no `\r`, no `\n` — just integers.

---

### Decoding Each Number

| Number | ASCII character | What it represents |
|---|---|---|
| 72 | `H` | letter H |
| 101 | `e` | letter e |
| 108 | `l` | letter l |
| 108 | `l` | letter l |
| 111 | `o` | letter o |
| **13** | — | **carriage return** (`\r`) |
| **10** | — | **line feed** (`\n`) |
| 87 | `W` | letter W |
| ... | ... | ... |

Notice: `13` and `10` are the ASCII codes for `\r` and `\n` — same lookup table you learned about with `ord()` and `chr()`!

```python
print(ord('\r'))    # → 13
print(ord('\n'))    # → 10
```

So `\r` and `\n` were **never magic control words** — they're just **byte number 13** and **byte number 10**, same as `72` is the byte for `H`.

---

### Why Does Python SHOW You `b'Hello\r\nWorld'` Then?

Because printing a raw list of numbers `[72, 101, 108, ...]` would be **unreadable** for humans. So Python's `bytes` type has a friendly `__repr__` (a "how do I show myself" method) that:

- Converts printable byte values (like 72) → their ASCII character (`H`)
- Converts non-printable byte values (like 13, 10) → an escape-style label (`\r`, `\n`)
- Wraps the whole thing with a `b'...'` prefix, as a visual reminder: *"this is bytes, not a real string!"*

```python
data = b'Hello\r\nWorld'

# 1-Python is showing you this...
b'Hello\r\nWorld'

# ...as a HUMAN-FRIENDLY VIEW of this:
[72, 101, 108, 108, 111, 13, 10, 87, 111, 114, 108, 100]
```

It's exactly like how `hex()` shows you `0xff` for the number `255` — a friendlier *display*, not a different *value*.

---

### The Critical Difference From a Real String

This is the whole point of the `b'...'` prefix — it's Python's warning label:

```python
text_version   = "Hello\r\nWorld"     # a str  — real characters
binary_version = b"Hello\r\nWorld"    # bytes  — raw numbers, DISPLAYED as if characters

print(type(text_version))     # → <class 'str'>
print(type(binary_version))   # → <class 'bytes'>

text_version == binary_version   # → False!  Different types entirely
```

They **look** the same on your screen because Python is being helpful with the display — but under the hood, one is Unicode text, the other is a raw list of 0–255 numbers that merely *happen* to correspond to printable ASCII when decoded.

---

### Connecting Back to "Nothing Was Converted"

Now the original sentence makes full sense:

> *"In binary mode, `\r` survives, nothing was converted"*

means: **the byte with value 13 (`\r`) is still sitting there, unmodified**, in position 6 of the raw byte sequence. Binary mode never looked at it, never touched it, never merged it with the following `\n` byte. It's just... byte number 13, exactly as stored on disk.

In **text mode**, Python actively **scans for the pair (13, 10)** and **deletes/merges** them into a single `\n` (byte 10 alone) when building your `str`. That scanning-and-merging is the "interpretation" that binary mode skips.

---

### The One-Sentence Summary

> `b'Hello\r\nWorld'` is **not text with escape characters typed literally** — it's Python's readable *display* of a raw sequence of bytes (`72, 101, 108, ..., 13, 10, ...`), where printable bytes are shown as their character and non-printable bytes (like 13 and 10) are shown using familiar `\r`/`\n` notation purely for your convenience. The `b` prefix is your signal: *"this is numbers, dressed up to be readable — not a real string."* 🎯

In the end of the mode string either the letter t or b can be appended. These stand for text mode and binary mode. If this letter is not given, the file type is text mode by default.

For binary mode the contents of the file are not interpreted in any way, and the read and write methods handle bytes. (A byte consists of 8 bits and can be used to represent a number in the range 0 to 255.)

In the text mode two interpretations happen

1. On Windows operating system the end of line in files is encoded by two characters. When the file is read these two charactes are converted to '\n' character. During writes to a file this conversion happens in the opposite direction.

2. One character is encoded in the file as one or more bytes. This conversion happens automatically during read and write operations. One common encoding between bytes and characters is utf-8. In this encoding, the Finnish character 'ä', for example, is encoded as the following sequence of bytes:

In [1]:
'ä'.encode('utf-8')

b'\xc3\xa4'

Above the two bytes were expressed as hexadecimals. In decimal notation they would be 195 and 164. (Both in the range from 0 to 255.)

In [2]:
list('ä'.encode('utf-8'))

[195, 164]

What is the utf-8 encoding of the letter 'a'?

In [3]:
'a'.encode('utf-8')

b'a'

In [4]:
list('a'.encode('utf-8'))

[97]

During this course we will only consider files containing text, so the default text mode is fine for us. But we might sometimes have to specify the encoding of a file, if it is not the usual ```utf-8```.